#### PATIENTS TABLE TRANSFORMATION

In [1]:
patients_df=spark.read.table("bronze_patients")
print("Total_Rows",patients_df.count())
print("Total_Columns",len(patients_df.columns))

StatementMeta(, 0ff82382-9ee9-46ec-9447-e207e09d28e9, 3, Finished, Available, Finished, False)

Total_Rows 25100
Total_Columns 8


In [2]:
print("\nColumns:")
print(patients_df.columns)

# Schema
print("\nSchema:")
patients_df.printSchema()

StatementMeta(, 0ff82382-9ee9-46ec-9447-e207e09d28e9, 4, Finished, Available, Finished, False)


Columns:
['PatientID', 'PatientName', 'DateOfBirth', 'Gender', 'City', 'State', 'InsuranceType', 'RegistrationDate']

Schema:
root
 |-- PatientID: string (nullable = true)
 |-- PatientName: string (nullable = true)
 |-- DateOfBirth: date (nullable = true)
 |-- Gender: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- InsuranceType: string (nullable = true)
 |-- RegistrationDate: date (nullable = true)



In [3]:
SELECT
    PatientID,
    COUNT(*) AS DuplicateCount
FROM bronze_patients
GROUP BY PatientID
HAVING COUNT(*) > 1;


StatementMeta(, 0ff82382-9ee9-46ec-9447-e207e09d28e9, 5, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 2 fields>

In [4]:
SELECT COUNT(*) AS DuplicatePatientIDs
FROM
(
    SELECT PatientID
    FROM bronze_patients
    GROUP BY PatientID
    HAVING COUNT(*) > 1
) AS d;

StatementMeta(, 0ff82382-9ee9-46ec-9447-e207e09d28e9, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
bronze_patients

Invalid DOB identify
 Future RegistrationDate identify
 Patient age calculate
 Age validity check
 Create cleaned silver_patients

In [12]:
%%sql

CREATE OR REPLACE TABLE bronze_patients AS
SELECT
    PatientID,
    PatientName,
    DateOfBirth,
    Gender,
    City,
    State,
    InsuranceType,
    RegistrationDate
FROM
(
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY PatientID
               ORDER BY RegistrationDate
           ) AS rn
    FROM bronze_patients
) t
WHERE rn = 1;

StatementMeta(, 0ff82382-9ee9-46ec-9447-e207e09d28e9, 14, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [13]:
SELECT count(*) AS Emp_count FROM bronze_patients
GROUP BY PatientID
HAVING COUNT(*)>1;

StatementMeta(, 0ff82382-9ee9-46ec-9447-e207e09d28e9, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

In [16]:
--NULL InsuranceType handle

SELECT * FROM bronze_patients
WHERE InsuranceType IS NULL;

StatementMeta(, 0ff82382-9ee9-46ec-9447-e207e09d28e9, 18, Finished, Available, Finished, False)

<Spark SQL result set with 250 rows and 8 fields>

In [17]:
SELECT
    PatientID,
    PatientName,
    DateOfBirth,
    Gender,
    City,
    State,
    COALESCE(InsuranceType, 'Unknown') AS InsuranceType,
    RegistrationDate
FROM bronze_patients;

StatementMeta(, 0ff82382-9ee9-46ec-9447-e207e09d28e9, 19, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 8 fields>

In [18]:
 --Gender values standardize

SELECT
    Gender,
    COUNT(*) AS Count
FROM bronze_patients
GROUP BY Gender
ORDER BY Count DESC;

StatementMeta(, 0ff82382-9ee9-46ec-9447-e207e09d28e9, 20, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 2 fields>

In [21]:
SELECT COUNT(*) AS NullGender
FROM bronze_patients
WHERE Gender IS NULL;

StatementMeta(, 0ff82382-9ee9-46ec-9447-e207e09d28e9, 23, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [22]:
SELECT DISTINCT Gender
FROM bronze_patients;

StatementMeta(, 0ff82382-9ee9-46ec-9447-e207e09d28e9, 24, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 1 fields>

In [27]:
--State values standardize

SELECT * FROM bronze_patients
WHERE State is unknown



StatementMeta(, 0ff82382-9ee9-46ec-9447-e207e09d28e9, 29, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 8 fields>

In [28]:
 --DateOfBirth datatype convert


DESCRIBE bronze_patients;

StatementMeta(, 0ff82382-9ee9-46ec-9447-e207e09d28e9, 30, Finished, Available, Finished, False)

<Spark SQL result set with 8 rows and 3 fields>

In [1]:
--Invalid DOB identify
SELECT * FROM bronze_patients
WHERE DateOfBirth > CURRENT_DATE();

StatementMeta(, 3060b490-5476-4014-8fc8-f42ee6e0f9b0, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 8 fields>

In [2]:
%%sql

SELECT
    PatientID,
    DateOfBirth,
    DATEDIFF(CURRENT_DATE(), DateOfBirth) / 365.25 AS Age
FROM bronze_patients
WHERE DATEDIFF(CURRENT_DATE(), DateOfBirth) / 365.25 > 90;

StatementMeta(, 3060b490-5476-4014-8fc8-f42ee6e0f9b0, 3, Finished, Available, Finished, False)

<Spark SQL result set with 281 rows and 3 fields>

In [3]:
%%sql

SELECT
    PatientID,
    DateOfBirth
FROM bronze_patients
WHERE DateOfBirth > CURRENT_DATE()
   OR DateOfBirth < ADD_MONTHS(CURRENT_DATE(), -12 * 100);

StatementMeta(, 3060b490-5476-4014-8fc8-f42ee6e0f9b0, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

In [ ]:

 Age validity check

In [4]:
%%sql

SELECT
    PatientID,
    RegistrationDate
FROM bronze_patients
WHERE RegistrationDate > CURRENT_DATE();

StatementMeta(, 3060b490-5476-4014-8fc8-f42ee6e0f9b0, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

In [8]:

%%sql

CREATE TABLE patient_age AS
SELECT
    PatientID,
    DateOfBirth,
    FLOOR(
        DATEDIFF(CURRENT_DATE(), DateOfBirth) / 365.25
    ) AS Age
FROM bronze_patients;

StatementMeta(, 3060b490-5476-4014-8fc8-f42ee6e0f9b0, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [9]:
%%sql

SELECT *
FROM patient_age
WHERE Age < 1
   OR Age > 90;

StatementMeta(, 3060b490-5476-4014-8fc8-f42ee6e0f9b0, 10, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 3 fields>

In [10]:
%%sql

SELECT COUNT(*) AS InvalidAgeCount
FROM patient_age
WHERE Age < 1
   OR Age > 90;

StatementMeta(, 3060b490-5476-4014-8fc8-f42ee6e0f9b0, 11, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [11]:
%%sql

SELECT
    Age,
    COUNT(*) AS PatientCount
FROM patient_age
GROUP BY Age
ORDER BY Age;

StatementMeta(, 3060b490-5476-4014-8fc8-f42ee6e0f9b0, 12, Finished, Available, Finished, False)

<Spark SQL result set with 90 rows and 2 fields>

In [12]:
%%sql

CREATE TABLE silver_patients AS

SELECT
    p.PatientID,
    p.PatientName,
    p.DateOfBirth,
    p.Gender,
    p.City,
    p.State,
    COALESCE(p.InsuranceType, 'Unknown') AS InsuranceType,
    p.RegistrationDate,
    a.Age

FROM bronze_patients p

LEFT JOIN patient_age a
    ON p.PatientID = a.PatientID;

StatementMeta(, 3060b490-5476-4014-8fc8-f42ee6e0f9b0, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [13]:
%%sql

SELECT *
FROM silver_patients
LIMIT 20;

StatementMeta(, 3060b490-5476-4014-8fc8-f42ee6e0f9b0, 14, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 9 fields>

In [14]:
%%sql

SELECT COUNT(*) AS TotalRows
FROM silver_patients;

StatementMeta(, 3060b490-5476-4014-8fc8-f42ee6e0f9b0, 15, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [3]:
%%sql
CREATE TABLE silver_hospitals AS
SELECT
    HospitalID,
    HospitalName,
    HospitalType,
    City,
    State,
    TotalBeds
FROM bronze_hospitals;

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
/*SELECT COUNT(*)AS Duplicates FROM bronze_departments
GROUP BY DepartmentID
HAVING COUNT(*)>1;


SELECT * FROM bronze_departments
WHERE DepartmentID IS NULL
*/


SELECT
    DepartmentName,
    COUNT(*) AS DepartmentCount
FROM bronze_departments
GROUP BY DepartmentName
ORDER BY DepartmentName;

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 8, Finished, Available, Finished, False)

<Spark SQL result set with 11 rows and 2 fields>

In [9]:
%%sql

/* SELECT d.HospitalID
FROM bronze_departments d
LEFT JOIN bronze_hospitals h
    ON d.HospitalID = h.HospitalID
WHERE h.HospitalID IS NULL;
*/
CREATE TABLE silver_departments AS
SELECT
    DepartmentID,
    TRIM(INITCAP(DepartmentName)) AS DepartmentName,
    HospitalID,
    DepartmentType
FROM bronze_departments;

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 10, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [12]:
SELECT
    DepartmentName,
    COUNT(*) AS DepartmentCount
FROM silver_departments
GROUP BY DepartmentName
ORDER BY DepartmentName;

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 13, Finished, Available, Finished, False)

<Spark SQL result set with 9 rows and 2 fields>

In [14]:
/* SELECT COUNT(*)AS Duplicates FROM bronze_doctors
GROUP BY DoctorID
HAVING COUNT(*)>1;
*/
SELECT * FROM bronze_doctors
WHERE DoctorID IS NULL;

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 6 fields>

In [15]:
DESC bronze_doctors

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 16, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 3 fields>

In [16]:
SELECT
    Specialization,
    COUNT(*) AS SpecializationCount
FROM bronze_doctors
GROUP BY Specialization
ORDER BY Specialization;

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 17, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 2 fields>

In [17]:
%%sql

SELECT d.DepartmentID
FROM bronze_doctors d
LEFT JOIN bronze_departments dept
    ON d.DepartmentID = dept.DepartmentID
WHERE dept.DepartmentID IS NULL;

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 18, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

In [18]:
%%sql

CREATE TABLE silver_doctors AS
SELECT
    DoctorID,
    TRIM(INITCAP(DoctorName)) AS DoctorName,
    DepartmentID,
    Specialization,
    EmploymentType,
    ExperienceYears
FROM bronze_doctors;

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

#### Appointments

In [21]:
/*SELECT COUNT(*) AS Duplicates FROM bronze_appointments
group by AppointmentID
HAVING COUNT(*)>1

SELECT * FROM bronze_appointments
WHERE AppointmentID IS NULL;
*/
DESC bronze_appointments

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 22, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 3 fields>

In [22]:
%%sql

SELECT *
FROM bronze_appointments
WHERE WaitingTimeMinutes < 0;

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 23, Finished, Available, Finished, False)

<Spark SQL result set with 50 rows and 7 fields>

In [25]:
/*SELECT AppointmentType , COUNT(*) AS AppointmentTypecount FROM bronze_appointments
GROUP BY AppointmentType
ORDER BY AppointmentType
*/
SELECT AppointmentStatus , COUNT(*) AS AppointmentStatuscount FROM bronze_appointments
GROUP BY AppointmentStatus
ORDER BY AppointmentStatus


StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 26, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 2 fields>

In [29]:
%%sql

/*SELECT a.PatientID
FROM bronze_appointments a
LEFT JOIN bronze_patients p
    ON a.PatientID = p.PatientID
WHERE p.PatientID IS NULL;
*/


SELECT a.DoctorID
FROM bronze_appointments a
LEFT JOIN bronze_doctors d
    ON a.DoctorID = d.DoctorID
WHERE d.DoctorID IS NULL;

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 31, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

In [33]:
%%sql

SELECT AppointmentStatus, COUNT(*) AS Count
FROM bronze_appointments
GROUP BY AppointmentStatus
ORDER BY Count DESC;

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 35, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 2 fields>

In [39]:
%%sql
CREATE TABLE silver_appointments AS
SELECT
    AppointmentID,
    PatientID,
    DoctorID,
    AppointmentDate,
    AppointmentType,
    AppointmentStatus,

    CASE
        WHEN WaitingTimeMinutes < 0 THEN NULL
        ELSE WaitingTimeMinutes
    END AS WaitingTimeMinutes,

    CASE
        WHEN WaitingTimeMinutes IS NULL OR WaitingTimeMinutes < 0 THEN 'Unknown'
        WHEN WaitingTimeMinutes <= 15 THEN 'Short'
        WHEN WaitingTimeMinutes <= 30 THEN 'Moderate'
        WHEN WaitingTimeMinutes <= 60 THEN 'Long'
        ELSE 'Very Long'
    END AS WaitingTimeCategory,

    CASE
        WHEN AppointmentStatus = 'No Show' THEN 'Yes'
        ELSE 'No'
    END AS IsNoShow,

    CASE
        WHEN AppointmentStatus = 'Cancelled' THEN 'Yes'
        ELSE 'No'
    END AS IsCancelled

FROM bronze_appointments;

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 41, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [40]:
SELECT * FROM silver_appointments;

StatementMeta(, 08a2961e-f906-4565-bf55-657ad93a4bcc, 42, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 10 fields>

In [ ]:
bronze_admissions
 Length of Stay calculate
 LOS validation
 LOS categories create
 Readmission-related data prepare
 Create silver_admissions

In [2]:
/*SELECT COUNT(*) AS Duplicate FROM bronze_admissions
GROUP BY AdmissionID 
HAVING COUNT(*)>1;
*/
SELECT * FROM bronze_admissions
WHERE AdmissionID IS NULL;

StatementMeta(, 9ad9acd4-a4c5-452b-9d66-84b83b135bf0, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 9 fields>

In [3]:
desc bronze_admissions

StatementMeta(, 9ad9acd4-a4c5-452b-9d66-84b83b135bf0, 4, Finished, Available, Finished, False)

<Spark SQL result set with 9 rows and 3 fields>

In [1]:
/* SELECT AdmissionType,COUNT(*)Admission_Count
FROM bronze_admissions
GROUP BY AdmissionType
ORDER BY AdmissionType
*/

SELECT COUNT(*) AS Duplicate FROM bronze_admissions
GROUP BY PatientID 
HAVING COUNT(*)>1;


StatementMeta(, d28e1346-8215-4448-b1e6-6953e330b661, 3, Finished, Available, Finished, True)

<Spark SQL result set with 1000 rows and 1 fields>

<Spark SQL result set with 0 rows and 9 fields>

In [2]:
%%sql
SELECT a.PatientID
FROM bronze_admissions a
LEFT JOIN bronze_patients p
    ON a.PatientID = p.PatientID
WHERE p.PatientID IS NULL;

StatementMeta(, d28e1346-8215-4448-b1e6-6953e330b661, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

In [4]:
SELECT COUNT(*) AS Duplicate FROM bronze_admissions
GROUP BY HospitalID 
HAVING COUNT(*)>1;

SELECT * FROM bronze_admissions
WHERE HospitalID IS NULL;

StatementMeta(, d28e1346-8215-4448-b1e6-6953e330b661, 8, Finished, Available, Finished, True)

<Spark SQL result set with 8 rows and 1 fields>

<Spark SQL result set with 0 rows and 9 fields>

In [5]:
%%sql
SELECT a.HospitalID
FROM bronze_admissions a
LEFT JOIN bronze_hospitals h
    ON a.HospitalID = h.HospitalID
WHERE h.HospitalID IS NULL;

StatementMeta(, d28e1346-8215-4448-b1e6-6953e330b661, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

In [7]:
SELECT a.DepartmentID
FROM bronze_admissions a
LEFT JOIN bronze_departments d
ON a.DepartmentID=d.DepartmentID
WHERE d.DepartmentID IS NULL;

StatementMeta(, d28e1346-8215-4448-b1e6-6953e330b661, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

In [9]:
SELECT * FROM bronze_admissions 
WHERE AdmissionDate > DischargeDate;

StatementMeta(, d28e1346-8215-4448-b1e6-6953e330b661, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 9 fields>

In [13]:
%%sql
SELECT
    MIN(DATEDIFF(DischargeDate, AdmissionDate)) AS Min_LOS,
    MAX(DATEDIFF(DischargeDate, AdmissionDate)) AS Max_LOS
FROM bronze_admissions
WHERE DischargeDate IS NOT NULL;

StatementMeta(, d28e1346-8215-4448-b1e6-6953e330b661, 17, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [17]:
%%sql 
CREATE TABLE silver_admissions AS 
SELECT 
    AdmissionID, 
    PatientID, 
    HospitalID, 
    DepartmentID, 
    AdmissionDate, 
    DischargeDate, 
    AdmissionType, 
    DischargeStatus, 
 
    CASE 
        WHEN DischargeDate IS NULL THEN NULL 
        ELSE DATEDIFF(DischargeDate, AdmissionDate) 
    END AS LengthOfStay, 
 
    CASE 
        WHEN DischargeDate IS NULL THEN 'Ongoing' 
        WHEN DATEDIFF(DischargeDate, AdmissionDate) <= 2 THEN 'Short Stay' 
        WHEN DATEDIFF(DischargeDate, AdmissionDate) <= 7 THEN 'Normal Stay' 
        WHEN DATEDIFF(DischargeDate, AdmissionDate) <= 14 THEN 'Long Stay' 
        ELSE 'Very Long Stay' 
    END AS LOSCategory, 
 
    LAG(AdmissionDate) OVER ( 
        PARTITION BY PatientID 
        ORDER BY AdmissionDate 
    ) AS PreviousAdmissionDate, 
 
    DATEDIFF( 
        AdmissionDate, 
        LAG(AdmissionDate) OVER ( 
            PARTITION BY PatientID 
            ORDER BY AdmissionDate 
        ) 
    ) AS DaysSincePreviousAdmission, 
 
    CASE 
        WHEN DATEDIFF( 
            AdmissionDate, 
            LAG(AdmissionDate) OVER ( 
                PARTITION BY PatientID 
                ORDER BY AdmissionDate 
            ) 
        ) <= 30 THEN 'Yes' 
        ELSE 'No' 
    END AS IsReadmission 
 
FROM bronze_admissions;

StatementMeta(, d28e1346-8215-4448-b1e6-6953e330b661, 21, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [18]:
SELECT * FROM silver_admissions

StatementMeta(, d28e1346-8215-4448-b1e6-6953e330b661, 22, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 13 fields>

#### Treatments

In [5]:
/*SELECT TreatmentID,COUNT(*)AS Duplicate
FROM bronze_treatments
GROUP BY TreatmentID
HAVING COUNT(*)>1;

SELECT * FROM bronze_treatments
WHERE TreatmentID IS NULL
*/
DESC bronze_treatments

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 6, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 3 fields>

In [7]:
/*SELECT t.patientID
FROM  bronze_treatments t 
LEFT JOIN bronze_patients p 
ON t.PatientID=p.patientID
WHERE p.PatientID IS NULL
*/
SELECT t.DoctorID
FROM  bronze_treatments t 
LEFT JOIN bronze_doctors d 
ON t.DoctorID=d.DoctorID
WHERE d.DoctorID IS NULL

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

In [8]:
%%sql
SELECT COUNT(*) AS Negative_TreatmentCost_Count
FROM bronze_treatments
WHERE TreatmentCost < 0;

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 9, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [9]:
SELECT * FROM bronze_treatments 
WHERE TreatmentCost IS NULL

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 10, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 6 fields>

In [10]:
%%sql
CREATE TABLE silver_treatments AS
SELECT
    TreatmentID,
    PatientID,
    DoctorID,
    TreatmentDate,

    TRIM(INITCAP(TreatmentType)) AS TreatmentType,

    CASE
        WHEN TreatmentCost < 0 THEN NULL
        ELSE TreatmentCost
    END AS TreatmentCost,

    CASE
        WHEN TreatmentCost < 0 THEN 'Yes'
        ELSE 'No'
    END AS IsNegativeCost,

    CASE
        WHEN TreatmentCost = 0 THEN 'Yes'
        ELSE 'No'
    END AS IsZeroCost,

    CASE
        WHEN TreatmentCost IS NULL THEN 'Unknown'
        WHEN TreatmentCost < 0 THEN 'Unknown'
        WHEN TreatmentCost = 0 THEN 'Free'
        WHEN TreatmentCost <= 1000 THEN 'Low'
        WHEN TreatmentCost <= 5000 THEN 'Medium'
        WHEN TreatmentCost <= 10000 THEN 'High'
        ELSE 'Very High'
    END AS CostCategory

FROM bronze_treatments;

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [14]:
SELECT HospitalID, DATE,COUNT(*) AS duplicate_date
FROM bronze_bed_utilization
GROUP BY HospitalID,DATE
HAVING COUNT(*)>1

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 3 fields>

In [23]:
/*SELECT * FROM bronze_bed_utilization
WHERE OccupancyRate IS NULL
*/
DESC bronze_bed_utilization

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 24, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 3 fields>

In [24]:
SELECT b.HospitalID
FROM bronze_bed_utilization b 
LEFT JOIN bronze_hospitals h 
ON b.HospitalID=h.HospitalID
WHERE h.HospitalID IS NULL

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 25, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

In [31]:
SELECT * FROM bronze_bed_utilization
WHERE TotalBeds <= OccupiedBeds

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 32, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 6 fields>

In [32]:
%%sql
CREATE TABLE silver_bed_utilization AS
SELECT
    Date,
    HospitalID,
    TotalBeds,
    OccupiedBeds,
    AvailableBeds,
    OccupancyRate,

    CASE
        WHEN OccupancyRate < 50 THEN 'Low'
        WHEN OccupancyRate < 75 THEN 'Moderate'
        WHEN OccupancyRate < 90 THEN 'High'
        ELSE 'Very High'
    END AS OccupancyCategory

FROM bronze_bed_utilization;

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 33, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
bronze_patient_feedback
 Rating range check
 Feedback category standardization
 Rating categories create
 Create silver_patient_feedback

In [35]:
/*SELECT FeedbackID, COUNT(*) AS duplicate_feedback_id
FROM bronze_patient_feedback
GROUP BY FeedbackID
HAVING COUNT(*)>1;

SELECT * FROM bronze_patient_feedback
WHERE FeedbackID IS NULL
*/
DESC bronze_patient_feedback


StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 36, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 3 fields>

In [39]:
SELECT * FROM bronze_patient_feedback
WHERE Rating IS NULL

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 40, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 6 fields>

In [40]:
SELECT f.PatientID
FROM bronze_patient_feedback f 
LEFT JOIN bronze_patients p 
ON f.PatientID =p.PatientID
WHERE p.PatientID IS NULL

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 41, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

In [41]:
SELECT f.DepartmentID
FROM bronze_patient_feedback f 
LEFT JOIN bronze_departments d  
ON f.DepartmentID =d.DepartmentID
WHERE d.DepartmentID IS NULL

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 42, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

In [42]:
%%sql
CREATE TABLE silver_patient_feedback AS
SELECT
    FeedbackID,
    PatientID,
    DepartmentID,
    Rating,

    TRIM(INITCAP(FeedbackCategory)) AS FeedbackCategory,

    FeedbackDate,

    CASE
        WHEN Rating IS NULL THEN 'Unknown'
        WHEN Rating <= 2 THEN 'Poor'
        WHEN Rating = 3 THEN 'Average'
        WHEN Rating = 4 THEN 'Good'
        WHEN Rating = 5 THEN 'Excellent'
        ELSE 'Invalid'
    END AS RatingCategory

FROM bronze_patient_feedback
WHERE Rating IS NULL
   OR Rating BETWEEN 1 AND 5;

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 43, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [48]:
/*SELECT ClaimID , COUNT(*) AS Duplicate_claim_ID
FROM bronze_insurance_claims 
GROUP BY ClaimID
HAVING COUNT(*)>1

SELECT ClaimID FROM bronze_insurance_claims
WHERE ClaimID IS NULL
*/
DESC bronze_insurance_claims

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 49, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 3 fields>

In [50]:
/*SELECT PatientID FROM bronze_insurance_claims
WHERE PatientID IS NULL
*/
SELECT c.PatientID
FROM bronze_insurance_claims c 
LEFT JOIN bronze_patients p
ON c.PatientID =p.PatientID
WHERE p.PatientID IS NULL

StatementMeta(, 440b73e9-8214-4486-ba3d-490cccfe6a17, 51, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

In [5]:
--Claim amount validation
SELECT * FROM bronze_insurance_claims
WHERE ClaimAmount < 0

StatementMeta(, 780bda56-74be-4c5b-bb3c-024be47be13a, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 7 fields>

In [9]:
%%sql
CREATE TABLE silver_insurance_claims AS
SELECT
    ClaimID,
    PatientID,
    ClaimDate,
    
    ClaimAmount,

    TRIM(INITCAP(ClaimStatus)) AS ClaimStatus,

    ApprovedAmount,
    
    CASE
        WHEN ClaimAmount IS NULL OR ClaimAmount = 0 THEN NULL
        ELSE ROUND((ApprovedAmount * 100.0) / ClaimAmount, 2)
    END AS ClaimApprovalPercentage,

    CASE
        WHEN ClaimAmount IS NULL  THEN 'Unknown'
        WHEN ClaimAmount = 0 THEN 'No Amount Claim'
        WHEN ClaimAmount <= 10000 THEN 'Low'
        WHEN ClaimAmount <= 50000 THEN 'Medium'
        WHEN ClaimAmount <= 100000 THEN 'High'
        ELSE 'Very High'
    END AS ClaimAmountCategory

FROM bronze_insurance_claims;

StatementMeta(, 780bda56-74be-4c5b-bb3c-024be47be13a, 10, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
Cross-table Data Quality


 Patient → Appointment validation
 Patient → Admission validation
 Patient → Treatment validation
 Patient → Feedback validation
 Patient → Claim validation
 Doctor → Appointment validation
 Doctor → Treatment validation
 Department → Doctor validation
 Department → Appointment validation
 Hospital → Department validation
 Hospital → Admission validation
 Hospital → Bed Utilization validation
 Treatment → Insurance Claim validation
 Admission → Treatment validation

In [3]:
SELECT * FROM silver_patients;

StatementMeta(, 53a71653-5c48-4edf-b259-3f66a2b7657c, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 9 fields>